In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)


In [ ]:
import pandas as pd

df = pd.read_csv("help_intent_dataset_v1_12000.csv")

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [ ]:
train_encodings = tokenizer(
    train_df["text"].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)
val_encodings = tokenizer(
    val_df["text"].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)

In [ ]:
import torch

train_labels = torch.tensor(
    train_df["label"].tolist(),
    dtype=torch.long
)

val_labels = torch.tensor(
    val_df["label"].tolist(),
    dtype=torch.long
)

In [ ]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(
    train_encodings["input_ids"],
    train_encodings["attention_mask"],
    train_labels
)

val_dataset = TensorDataset(
    val_encodings["input_ids"],
    val_encodings["attention_mask"],
    val_labels
)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)
for batch in train_loader:
    input_ids = batch[0]
    attention_mask = batch[1]
    labels = batch[2]
    
    optimizer.zero_grad()
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )
    loss = loss_fn(outputs.logits, labels)
    
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()

In [ ]:
correct = 0
total = 0

with torch.no_grad():

    for batch in val_loader:
        input_ids = batch[0]
        attention_mask = batch[1]
        labels = batch[2]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print("Validation Accuracy:", accuracy)

In [ ]:
texts = [
    "what is webkit",
    "how does webkit work",
    "I'm just reading about webkit",
    "hmm this code looks weird",
    "can you explain this error",
    "I think I understand it now",
    "why is my CSS not working",
    "okay let me try this myself"
]
inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"
)
with torch.no_grad():
    outputs = model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"]
    )

    probabilities = torch.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(outputs.logits, dim=1)

for text, prediction, probability in zip(
    texts,
    predictions,
    probabilities
):
    print("Text:", text)
    print("Prediction:", prediction.item())
    print("P(should respond):", probability[1].item())
    print()

In [ ]:
import os

save_path = "./should_ai_respond_model"

os.makedirs(save_path, exist_ok=True)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to:", save_path)

## Loading Saved models- testing

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

loaded_tokenizer = AutoTokenizer.from_pretrained(
    "./should_ai_respond_model"
)

loaded_model = AutoModelForSequenceClassification.from_pretrained(
    "./should_ai_respond_model"
)

loaded_model.eval()

print("Model loaded successfully")
text = "what is webkit"

inputs = loaded_tokenizer(
    text,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = loaded_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

print("P(do not respond):", probabilities[0, 0].item())
print("P(should respond):", probabilities[0, 1].item())

In [ ]:
def should_respond(text, threshold=0.80):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)

    probability = probabilities[0, 1].item()

    return probability >= threshold, probability

texts = [
    "what is webkit",
    "I'm just reading about webkit",
    "can you explain this error",
    "I think I understand it now",
    "okay let me try this myself"
]

for text in texts:
    respond, probability = should_respond(text)

    print(
        text,
        "=>",
        "RESPOND" if respond else "SILENT",
        f"({probability:.3f})"
    )

## Quantization 

In [ ]:
import onnx

model_path = "./should_ai_respond.onnx"

onnx_model = onnx.load(model_path)

# Remove existing inferred value information.
onnx_model.graph.ClearField("value_info")

clean_path = "./should_ai_respond_clean.onnx"

onnx.save(
    onnx_model,
    clean_path
)

print("Clean ONNX model created")

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    "./should_ai_respond_clean.onnx",
    providers=["CPUExecutionProvider"]
)

print("Clean ONNX model loaded successfully")

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="./should_ai_respond_clean.onnx",
    model_output="./should_ai_respond_int8.onnx",
    weight_type=QuantType.QInt8
)

print("INT8 quantization completed")

In [ ]:
import os

int8_size = os.path.getsize(
    "./should_ai_respond_int8.onnx"
) / (1024 ** 2)

print(f"INT8 ONNX size: {int8_size:.2f} MB")


In [ ]:
import onnxruntime as ort
import numpy as np

fp32_session = ort.InferenceSession(
    "./should_ai_respond.onnx",
    providers=["CPUExecutionProvider"]
)

int8_session = ort.InferenceSession(
    "./should_ai_respond_int8.onnx",
    providers=["CPUExecutionProvider"]
)

print("Both models loaded successfully")

In [ ]:
text = "webkit anty enti"

inputs = tokenizer(
    text,
    return_tensors="np",
    truncation=True
)

onnx_inputs = {
    "input_ids": inputs["input_ids"],
    "attention_mask": inputs["attention_mask"]
}

fp32_logits = fp32_session.run(
    None,
    onnx_inputs
)[0]

int8_logits = int8_session.run(
    None,
    onnx_inputs
)[0]

In [ ]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / exp_x.sum(axis=1, keepdims=True)


fp32_probs = softmax(fp32_logits)
int8_probs = softmax(int8_logits)

print("FP32 probabilities:", fp32_probs)
print("INT8 probabilities:", int8_probs)

In [ ]:
fp32_prediction = np.argmax(fp32_probs, axis=1)[0]
int8_prediction = np.argmax(int8_probs, axis=1)[0]

print("FP32 prediction:", fp32_prediction)
print("INT8 prediction:", int8_prediction)